In [ ]:
# ============================================================================#          FRAUD DETECTION - COMPLETE MODEL COMPARISON#          Comparing: Logistic Regression, Random Forest, SVM, XGBoost, Naive Bayes#          Baseline: ANN (Sensitivity 81.78%, Precision 70.18%, F1 75.49%)# ============================================================================println("🚀 E-Commerce Fraud Detection - Model Comparison Analysis")println("="^70)using Pkgprintln("\n📦 Installing required packages (this may take several minutes)...")packages = ["CSV", "DataFrames", "Statistics", "Random", "Dates", "StatsBase",            "MLJ", "MLJLinearModels", "MLJDecisionTreeInterface",             "MLJLIBSVMInterface", "XGBoost", "Plots", "StatsPlots"]for pkg in packages    try        Pkg.add(pkg)    catch e        println("  ⚠️  Could not install $pkg")    endendprintln("\n📚 Loading packages...")using CSV, DataFrames, Statistics, Random, Dates, StatsBaseusing MLJusing Plots, StatsPlotsprintln("✅ Packages loaded successfully!")

In [ ]:
# ============================================================================#              LOAD UTILITY FUNCTIONS# ============================================================================# Load existing utility functions from utils.jlinclude("utils.jl")# Load model comparison specific utilitiesinclude("model_utils.jl")println("✅ Utility functions loaded!")

In [ ]:
# ============================================================================#              DATA LOADING & PREPROCESSING - TWO DATASETS#              1. Balanced (50/50) - for comparison with baseline#              2. Realistic (unbalanced) - respecting natural fraud rate# ============================================================================# Same preprocessing function as miglior compromesso.ipynbfunction preprocess_data_enhanced(dataframe)    data = copy(dataframe)        println("\n🔧 Creating Enhanced Features...")        # TIME FEATURES    if "Transaction Date" in names(data)        try            if eltype(data[!, "Transaction Date"]) <: AbstractString                 data.ParsedDate = DateTime.(data[!, "Transaction Date"], dateformat"y-m-d H:M:S")             else                 data.ParsedDate = data[!, "Transaction Date"]            end            data.Hour = hour.(data.ParsedDate)            data.Is_Night = [h < 6 ? 1.0 : 0.0 for h in data.Hour]            data.Is_Weekend = [dayofweek(d) in [6,7] ? 1.0 : 0.0 for d in data.ParsedDate]            data.Hour_Risk = [h in [0,1,2,3,4,5,23] ? 1.0 : 0.0 for h in data.Hour]            data.Is_Early_Morning = [h in [2,3,4,5] ? 1.0 : 0.0 for h in data.Hour]            println("  ✓ Time features created")        catch e            println("  ⚠ Error: $e")        end    end    # IMPUTATION    for col in ["Transaction Amount", "Quantity", "Customer Age", "Account Age Days"]        if col in names(data) && any(ismissing, data[!, col])            median_val = median(skipmissing(data[!, col]))            replace!(data[!, col], missing => median_val)        end    end        # RISK FEATURES    if "Transaction Amount" in names(data) && "Account Age Days" in names(data)        data.Amount_per_AccountAge = data[!, "Transaction Amount"] ./ (data[!, "Account Age Days"] .+ 1.0)    end        if "Transaction Amount" in names(data)        p95 = quantile(data[!, "Transaction Amount"], 0.95)        p99 = quantile(data[!, "Transaction Amount"], 0.99)        data.High_Value_Flag = [amt > p95 ? 1.0 : 0.0 for amt in data[!, "Transaction Amount"]]        data.Very_High_Value_Flag = [amt > p99 ? 1.0 : 0.0 for amt in data[!, "Transaction Amount"]]    end        if "Quantity" in names(data)        data.High_Qty_Flag = [q > 5 ? 1.0 : 0.0 for q in data[!, "Quantity"]]        data.Very_High_Qty_Flag = [q > 10 ? 1.0 : 0.0 for q in data[!, "Quantity"]]                if "Transaction Amount" in names(data)            data.Unit_Price = data[!, "Transaction Amount"] ./ (data[!, "Quantity"] .+ 0.1)            p95_unit = quantile(data.Unit_Price, 0.95)            data.High_Unit_Price_Flag = [up > p95_unit ? 1.0 : 0.0 for up in data.Unit_Price]        end    end        if "Account Age Days" in names(data)        data.New_Account_Flag = [age < 30 ? 1.0 : 0.0 for age in data[!, "Account Age Days"]]        data.Very_New_Account_Flag = [age < 7 ? 1.0 : 0.0 for age in data[!, "Account Age Days"]]    end        if "Customer Age" in names(data)        data.Young_Customer_Flag = [age < 25 ? 1.0 : 0.0 for age in data[!, "Customer Age"]]        data.Senior_Customer_Flag = [age > 65 ? 1.0 : 0.0 for age in data[!, "Customer Age"]]    end        # COMBINED RISK SCORE    risk_cols = []    for col in ["High_Value_Flag", "High_Qty_Flag", "New_Account_Flag",                 "Hour_Risk", "Is_Night", "Young_Customer_Flag"]        if col in names(data)            push!(risk_cols, col)        end    end        if !isempty(risk_cols)        data.Risk_Score = sum([data[!, col] for col in risk_cols])    end        # CLEANUP    cols_to_drop = ["Transaction ID", "Customer ID", "Transaction Date", "ParsedDate",                     "IP Address", "Shipping Address", "Billing Address", "Customer Location"]    select!(data, Not(intersect(names(data), cols_to_drop)))        # ONE-HOT ENCODING    categorical_cols = ["Payment Method", "Product Category", "Device Used"]    input_df = data[:, setdiff(names(data), categorical_cols)]        for col in categorical_cols        if col in names(data)            encoded_matrix = oneHotEncoding(data[!, col])            new_col_names = ["$(col)_$(i)" for i in 1:size(encoded_matrix, 2)]            encoded_df = DataFrame(encoded_matrix, new_col_names)            input_df = hcat(input_df, encoded_df)        end    end        println("✅ Feature engineering completed!")    return input_dfend# LOAD DATAconst DATA_PATH = "Fraudulent_E-Commerce_Transaction_Data_merge.csv"if !isfile(DATA_PATH)    error("Dataset not found! Please download from Kaggle and save as: $DATA_PATH")endprintln("\n📂 Loading full dataset...")df_full = CSV.read(DATA_PATH, DataFrame)target_col = "Is Fraudulent"println("\n" * "="^70)println("PREPARING TWO DATASETS FOR COMPARISON")println("="^70)# ========== DATASET 1: BALANCED (50/50) ==========println("\n📊 Dataset 1: BALANCED (50/50 - Artificial Balance)")println("   Purpose: Comparison with baseline ANN and model training")fraud_rows_all = df_full[df_full[:, target_col] .== 1, :]n_fraud = size(fraud_rows_all, 1)println("   Total frauds available: $n_fraud")non_fraud_rows_all = df_full[df_full[:, target_col] .== 0, :]n_non_fraud = size(non_fraud_rows_all, 1)println("   Total legitimate transactions: $n_non_fraud")println("   Natural fraud rate: $(round(n_fraud/(n_fraud+n_non_fraud)*100, digits=2))%")# Sample equal number of non-fraudsRandom.seed!(42)non_fraud_sample_balanced = non_fraud_rows_all[shuffle(1:n_non_fraud)[1:n_fraud], :]df_balanced = vcat(fraud_rows_all, non_fraud_sample_balanced)df_balanced = df_balanced[shuffle(1:size(df_balanced, 1)), :] println("   ✅ Balanced dataset: $(size(df_balanced, 1)) samples (50% fraud / 50% legit)")# ========== DATASET 2: REALISTIC (UNBALANCED) ==========println("\n📊 Dataset 2: REALISTIC (Unbalanced - Natural Fraud Rate)")println("   Purpose: Real-world scenario simulation")# Take a reasonable sample size (e.g., 20,000 samples) maintaining natural proportionsample_size_realistic = min(20000, size(df_full, 1))Random.seed!(42)sample_indices = shuffle(1:size(df_full, 1))[1:sample_size_realistic]df_realistic = df_full[sample_indices, :]n_fraud_realistic = sum(df_realistic[:, target_col] .== 1)fraud_rate_realistic = n_fraud_realistic / sample_size_realistic * 100println("   Sample size: $sample_size_realistic")println("   Frauds: $n_fraud_realistic")println("   Fraud rate: $(round(fraud_rate_realistic, digits=2))%")println("   ✅ Realistic dataset prepared")# ========== PREPROCESS BOTH DATASETS ==========println("\n🔧 Preprocessing datasets...")println("\n--- Processing Balanced Dataset ---")df_processed_balanced = preprocess_data_enhanced(df_balanced)input_cols_balanced = setdiff(names(df_processed_balanced), [target_col])inputs_balanced = Matrix{Float64}(df_processed_balanced[:, input_cols_balanced])targets_balanced = Bool.(vec(df_processed_balanced[:, target_col]))println("\n--- Processing Realistic Dataset ---")df_processed_realistic = preprocess_data_enhanced(df_realistic)input_cols_realistic = setdiff(names(df_processed_realistic), [target_col])inputs_realistic = Matrix{Float64}(df_processed_realistic[:, input_cols_realistic])targets_realistic = Bool.(vec(df_processed_realistic[:, target_col]))println("\n" * "="^70)println("📊 DATASETS READY")println("="^70)println("\nBalanced Dataset:")println("   Features: $(size(inputs_balanced, 2))")println("   Samples: $(size(inputs_balanced, 1))")println("   Fraud rate: $(round(mean(targets_balanced)*100, digits=1))%")println("\nRealistic Dataset:")println("   Features: $(size(inputs_realistic, 2))")println("   Samples: $(size(inputs_realistic, 1))")println("   Fraud rate: $(round(mean(targets_realistic)*100, digits=1))%")# Create CV indices for both datasetsRandom.seed!(42)cv_indices_balanced = crossvalidation(targets_balanced, 3)cv_indices_realistic = crossvalidation(targets_realistic, 3)println("\n✅ Both datasets ready for modeling!")

In [ ]:
# ============================================================================#              MODEL 1: LOGISTIC REGRESSION# ============================================================================println("\n🔄 Training Logistic Regression with 3-fold CV...")LogisticClassifier = @load LogisticClassifier pkg=MLJLinearModelslr_metrics = []for fold in 1:3    println("  Fold $fold/3...")        test_mask = cv_indices .== fold    train_mask = .!test_mask        X_train = inputs[train_mask, :]    y_train = targets_bool[train_mask]    X_test = inputs[test_mask, :]    y_test = targets_bool[test_mask]        # Normalize (required for LogReg)    norm_params = calculateMinMaxNormalizationParameters(X_train)    X_train_norm = normalizeMinMax(X_train, norm_params)    X_test_norm = normalizeMinMax(X_test, norm_params)        # Tune lambda (L2 regularization)    best_sens = 0.0    best_pred = nothing        for lambda in [0.001, 0.01, 0.1, 1.0, 10.0]        model = LogisticClassifier(lambda=lambda)        mach = machine(model, X_train_norm, y_train)        MLJ.fit!(mach, verbosity=0)                y_pred = MLJ.predict_mode(mach, X_test_norm)        m = calculate_comprehensive_metrics(y_test, y_pred)                if m["sensitivity"] > best_sens            best_sens = m["sensitivity"]            best_pred = y_pred        end    end        push!(lr_metrics, calculate_comprehensive_metrics(y_test, best_pred))endlr_avg = aggregate_cv_metrics(lr_metrics)print_model_results("LOGISTIC REGRESSION", lr_avg)

In [ ]:
# ============================================================================#              TRAINING WRAPPER FOR DUAL DATASET COMPARISON# ============================================================================function train_model_on_both_datasets(model_name, train_function)    """    Train a model on both balanced and realistic datasets    Returns dictionaries with results for each dataset    """    println("\n" * "="^70)    println("🔄 Training $model_name on BOTH datasets")    println("="^70)        results = Dict()        # Train on BALANCED dataset    println("\n📊 Dataset 1: BALANCED (50/50)")    println("-"^70)    results["balanced"] = train_function(inputs_balanced, targets_balanced, cv_indices_balanced)        # Train on REALISTIC dataset      println("\n📊 Dataset 2: REALISTIC (Unbalanced)")    println("-"^70)    results["realistic"] = train_function(inputs_realistic, targets_realistic, cv_indices_realistic)        return resultsendprintln("✅ Training wrapper ready!")

In [ ]:
# ============================================================================#              MODEL 1: LOGISTIC REGRESSION (Both Datasets)# ============================================================================println("\n🔄 Training Logistic Regression on BOTH datasets...")LogisticClassifier = @load LogisticClassifier pkg=MLJLinearModelsfunction train_logistic_regression(inputs, targets, cv_indices)    metrics_list = []        for fold in 1:3        println("  Fold $fold/3...")                test_mask = cv_indices .== fold        train_mask = .!test_mask                X_train = inputs[train_mask, :]        y_train = targets[train_mask]        X_test = inputs[test_mask, :]        y_test = targets[test_mask]                # Normalize        norm_params = calculateMinMaxNormalizationParameters(X_train)        X_train_norm = normalizeMinMax(X_train, norm_params)        X_test_norm = normalizeMinMax(X_test, norm_params)                # Tune lambda        best_sens = 0.0        best_pred = nothing                for lambda in [0.001, 0.01, 0.1, 1.0, 10.0]            model = LogisticClassifier(lambda=lambda)            mach = machine(model, X_train_norm, y_train)            MLJ.fit!(mach, verbosity=0)                        y_pred = MLJ.predict_mode(mach, X_test_norm)            m = calculate_comprehensive_metrics(y_test, y_pred)                        if m["sensitivity"] > best_sens                best_sens = m["sensitivity"]                best_pred = y_pred            end        end                push!(metrics_list, calculate_comprehensive_metrics(y_test, best_pred))    end        return aggregate_cv_metrics(metrics_list)end# Train on both datasetslr_results = train_model_on_both_datasets("Logistic Regression", train_logistic_regression)# Print results for bothprintln("\n" * "="^70)println("📊 LOGISTIC REGRESSION - RESULTS COMPARISON")println("="^70)println("\n🔵 BALANCED Dataset (50/50):")print_model_results("Logistic Regression - Balanced", lr_results["balanced"])println("\n🟢 REALISTIC Dataset (Unbalanced):")print_model_results("Logistic Regression - Realistic", lr_results["realistic"])

In [ ]:
# ============================================================================#              MODEL 2: RANDOM FOREST (Both Datasets)# ============================================================================println("\n🔄 Training Random Forest on BOTH datasets...")RandomForestClassifier = @load RandomForestClassifier pkg=DecisionTreefunction train_random_forest(inputs, targets, cv_indices)    metrics_list = []        for fold in 1:3        println("  Fold $fold/3...")                test_mask = cv_indices .== fold        train_mask = .!test_mask                X_train = inputs[train_mask, :]        y_train = targets[train_mask]        X_test = inputs[test_mask, :]        y_test = targets[test_mask]                best_sens = 0.0        best_pred = nothing                for n_trees in [100, 200]            for max_d in [10, 20, -1]                model = RandomForestClassifier(n_trees=n_trees, max_depth=max_d)                mach = machine(model, X_train, y_train)                MLJ.fit!(mach, verbosity=0)                                y_pred = MLJ.predict_mode(mach, X_test)                m = calculate_comprehensive_metrics(y_test, y_pred)                                if m["sensitivity"] > best_sens                    best_sens = m["sensitivity"]                    best_pred = y_pred                end            end        end                push!(metrics_list, calculate_comprehensive_metrics(y_test, best_pred))    end        return aggregate_cv_metrics(metrics_list)endrf_results = train_model_on_both_datasets("Random Forest", train_random_forest)println("\n" * "="^70)println("📊 RANDOM FOREST - RESULTS COMPARISON")println("="^70)println("\n🔵 BALANCED Dataset (50/50):")print_model_results("Random Forest - Balanced", rf_results["balanced"])println("\n🟢 REALISTIC Dataset (Unbalanced):")print_model_results("Random Forest - Realistic", rf_results["realistic"])

In [ ]:
# ============================================================================#              MODEL 3: SUPPORT VECTOR MACHINE (Both Datasets)# ============================================================================println("\n🔄 Training SVM on BOTH datasets...")SVC = @load SVC pkg=LIBSVMfunction train_svm(inputs, targets, cv_indices)    metrics_list = []        for fold in 1:3        println("  Fold $fold/3...")                test_mask = cv_indices .== fold        train_mask = .!test_mask                X_train = inputs[train_mask, :]        y_train = targets[train_mask]        X_test = inputs[test_mask, :]        y_test = targets[test_mask]                # Normalize        norm_params = calculateMinMaxNormalizationParameters(X_train)        X_train_norm = normalizeMinMax(X_train, norm_params)        X_test_norm = normalizeMinMax(X_test, norm_params)                best_sens = 0.0        best_pred = nothing                for cost in [0.1, 1.0, 10.0]            try                model = SVC(kernel="rbf", cost=cost)                mach = machine(model, X_train_norm, y_train)                MLJ.fit!(mach, verbosity=0)                                y_pred = MLJ.predict_mode(mach, X_test_norm)                m = calculate_comprehensive_metrics(y_test, y_pred)                                if m["sensitivity"] > best_sens                    best_sens = m["sensitivity"]                    best_pred = y_pred                end            catch e                println("    ⚠️  SVM failed: $e")            end        end                if best_pred !== nothing            push!(metrics_list, calculate_comprehensive_metrics(y_test, best_pred))        end    end        return !isempty(metrics_list) ? aggregate_cv_metrics(metrics_list) : nothingendsvm_results_balanced = train_svm(inputs_balanced, targets_balanced, cv_indices_balanced)svm_results_realistic = train_svm(inputs_realistic, targets_realistic, cv_indices_realistic)svm_results = Dict("balanced" => svm_results_balanced, "realistic" => svm_results_realistic)println("\n" * "="^70)println("📊 SVM - RESULTS COMPARISON")println("="^70)if svm_results["balanced"] !== nothing    println("\n🔵 BALANCED Dataset (50/50):")    print_model_results("SVM - Balanced", svm_results["balanced"])else    println("\n🔵 BALANCED Dataset: ⚠️  Training failed")endif svm_results["realistic"] !== nothing    println("\n🟢 REALISTIC Dataset (Unbalanced):")    print_model_results("SVM - Realistic", svm_results["realistic"])else    println("\n🟢 REALISTIC Dataset: ⚠️  Training failed")end

In [ ]:
# ============================================================================#              MODEL 4: XGBOOST (Both Datasets)# ============================================================================println("\n🔄 Training XGBoost on BOTH datasets...")using XGBoostXGBoostClassifier = @load XGBoostClassifier pkg=XGBoostfunction train_xgboost(inputs, targets, cv_indices)    metrics_list = []        for fold in 1:3        println("  Fold $fold/3...")                test_mask = cv_indices .== fold        train_mask = .!test_mask                X_train = inputs[train_mask, :]        y_train = targets[train_mask]        X_test = inputs[test_mask, :]        y_test = targets[test_mask]                best_sens = 0.0        best_pred = nothing                for eta in [0.01, 0.05, 0.1]            for depth in [3, 5, 7]                for rounds in [100, 200]                    try                        model = XGBoostClassifier(                            eta=eta,                            max_depth=depth,                            num_round=rounds,                            objective="binary:logistic"                        )                        mach = machine(model, X_train, y_train)                        MLJ.fit!(mach, verbosity=0)                                                y_pred = MLJ.predict_mode(mach, X_test)                        m = calculate_comprehensive_metrics(y_test, y_pred)                                                if m["sensitivity"] > best_sens                            best_sens = m["sensitivity"]                            best_pred = y_pred                        end                    catch e                        # Silently continue                    end                end            end        end                if best_pred !== nothing            push!(metrics_list, calculate_comprehensive_metrics(y_test, best_pred))        end    end        return !isempty(metrics_list) ? aggregate_cv_metrics(metrics_list) : nothingendxgb_results_balanced = train_xgboost(inputs_balanced, targets_balanced, cv_indices_balanced)xgb_results_realistic = train_xgboost(inputs_realistic, targets_realistic, cv_indices_realistic)xgb_results = Dict("balanced" => xgb_results_balanced, "realistic" => xgb_results_realistic)println("\n" * "="^70)println("📊 XGBOOST - RESULTS COMPARISON")println("="^70)if xgb_results["balanced"] !== nothing    println("\n🔵 BALANCED Dataset (50/50):")    print_model_results("XGBoost - Balanced", xgb_results["balanced"])else    println("\n🔵 BALANCED Dataset: ⚠️  Training failed")endif xgb_results["realistic"] !== nothing    println("\n🟢 REALISTIC Dataset (Unbalanced):")    print_model_results("XGBoost - Realistic", xgb_results["realistic"])else    println("\n🟢 REALISTIC Dataset: ⚠️  Training failed")end

In [ ]:
# ============================================================================#              MODEL 5: NAIVE BAYES (Both Datasets)# ============================================================================println("\n🔄 Training Naive Bayes on BOTH datasets...")nb_results = Dict("balanced" => nothing, "realistic" => nothing)try    using NaiveBayes        function train_naive_bayes(inputs, targets, cv_indices)        metrics_list = []                for fold in 1:3            println("  Fold $fold/3...")                        test_mask = cv_indices .== fold            train_mask = .!test_mask                        X_train = inputs[train_mask, :]'  # NaiveBayes expects features x samples            y_train = targets[train_mask]            X_test = inputs[test_mask, :]'            y_test = targets[test_mask]                        model = GaussianNB(unique(y_train))            NaiveBayes.fit(model, X_train, y_train)                        y_pred = NaiveBayes.predict(model, X_test)                        push!(metrics_list, calculate_comprehensive_metrics(y_test, y_pred))        end                return aggregate_cv_metrics(metrics_list)    end        nb_results["balanced"] = train_naive_bayes(inputs_balanced, targets_balanced, cv_indices_balanced)    nb_results["realistic"] = train_naive_bayes(inputs_realistic, targets_realistic, cv_indices_realistic)        println("\n" * "="^70)    println("📊 NAIVE BAYES - RESULTS COMPARISON")    println("="^70)        println("\n🔵 BALANCED Dataset (50/50):")    print_model_results("Naive Bayes - Balanced", nb_results["balanced"])        println("\n🟢 REALISTIC Dataset (Unbalanced):")    print_model_results("Naive Bayes - Realistic", nb_results["realistic"])    catch e    println("⚠️  Naive Bayes not available: $e")end

In [ ]:
# ============================================================================#              FINAL COMPARISON & ANALYSIS - BOTH DATASETS# ============================================================================println("\n\n" * "="^80)println("📊 COMPLETE MODEL COMPARISON - TWO DATASETS")println("="^80)# ANN baseline (from miglior compromesso.ipynb - trained on balanced data)ann_baseline = Dict(    "accuracy" => 0.7341,    "sensitivity" => 0.8178,    "specificity" => 0.6504,    "precision" => 0.7018,    "f1" => 0.7549,    "f2" => 0.7826,    "TP" => 61385,    "TN" => 48766,    "FP" => 26241,    "FN" => 13675)# ========== BALANCED DATASET COMPARISON ==========println("\n" * "="^80)println("🔵 COMPARISON ON BALANCED DATASET (50/50)")println("="^80)balanced_models = Dict("ANN (Baseline)" => ann_baseline)if @isdefined(lr_results) && haskey(lr_results, "balanced")    balanced_models["Logistic Regression"] = lr_results["balanced"]endif @isdefined(rf_results) && haskey(rf_results, "balanced")    balanced_models["Random Forest"] = rf_results["balanced"]endif @isdefined(svm_results) && haskey(svm_results, "balanced") && svm_results["balanced"] !== nothing    balanced_models["SVM"] = svm_results["balanced"]endif @isdefined(xgb_results) && haskey(xgb_results, "balanced") && xgb_results["balanced"] !== nothing    balanced_models["XGBoost"] = xgb_results["balanced"]endif @isdefined(nb_results) && haskey(nb_results, "balanced") && nb_results["balanced"] !== nothing    balanced_models["Naive Bayes"] = nb_results["balanced"]endcreate_comparison_table(balanced_models)print_business_analysis(balanced_models)# ========== REALISTIC DATASET COMPARISON ==========println("\n" * "="^80)println("🟢 COMPARISON ON REALISTIC DATASET (Unbalanced)")println("="^80)realistic_models = Dict()if @isdefined(lr_results) && haskey(lr_results, "realistic")    realistic_models["Logistic Regression"] = lr_results["realistic"]endif @isdefined(rf_results) && haskey(rf_results, "realistic")    realistic_models["Random Forest"] = rf_results["realistic"]endif @isdefined(svm_results) && haskey(svm_results, "realistic") && svm_results["realistic"] !== nothing    realistic_models["SVM"] = svm_results["realistic"]endif @isdefined(xgb_results) && haskey(xgb_results, "realistic") && xgb_results["realistic"] !== nothing    realistic_models["XGBoost"] = xgb_results["realistic"]endif @isdefined(nb_results) && haskey(nb_results, "realistic") && nb_results["realistic"] !== nothing    realistic_models["Naive Bayes"] = nb_results["realistic"]endcreate_comparison_table(realistic_models)print_business_analysis(realistic_models)# ========== RECOMMENDATIONS ==========println("\n" * "="^80)println("🏆 RECOMMENDATIONS")println("="^80)# Find best model on balanced datasetbest_model_balanced = ""best_sensitivity_balanced = 0.0for (name, metrics) in balanced_models    if metrics["sensitivity"] > best_sensitivity_balanced        best_sensitivity_balanced = metrics["sensitivity"]        best_model_balanced = name    endendprintln("\n1. BEST FOR BALANCED DATA (Training/Comparison):")println("   ⭐ $best_model_balanced")println("   Sensitivity: $(round(best_sensitivity_balanced*100, digits=2))%")# Find best model on realistic datasetif !isempty(realistic_models)    best_model_realistic = ""    best_sensitivity_realistic = 0.0    for (name, metrics) in realistic_models        if metrics["sensitivity"] > best_sensitivity_realistic            best_sensitivity_realistic = metrics["sensitivity"]            best_model_realistic = name        end    end        println("\n2. BEST FOR REALISTIC DATA (Production Scenario):")    println("   ⭐ $best_model_realistic")    println("   Sensitivity: $(round(best_sensitivity_realistic*100, digits=2))%")    println("   Note: Lower precision expected due to class imbalance")endprintln("\n3. KEY INSIGHTS:")println("   • Balanced dataset helps models learn patterns equally")println("   • Realistic dataset shows true production performance")println("   • Higher false positives expected with imbalanced data")println("   • Sensitivity remains primary metric for fraud detection")println("\n4. NEXT STEPS:")println("   ✓ Use balanced data for initial model training")println("   ✓ Validate on realistic/imbalanced data before deployment")println("   ✓ Consider threshold tuning for production")println("   ✓ Monitor both precision and recall in production")println("\n" * "="^80)println("✅ DUAL DATASET COMPARISON COMPLETE!")println("="^80)

In [ ]:
# ============================================================================#              VISUALIZATIONS - COMPARISON ACROSS DATASETS# ============================================================================println("\n📊 Creating comparison visualizations...")# Prepare data for BALANCED datasetif !isempty(balanced_models)    model_names_bal = collect(keys(balanced_models))    sensitivities_bal = [balanced_models[m]["sensitivity"] for m in model_names_bal]    precisions_bal = [balanced_models[m]["precision"] for m in model_names_bal]    f1_scores_bal = [balanced_models[m]["f1"] for m in model_names_bal]        p_bal = groupedbar(        [sensitivities_bal precisions_bal f1_scores_bal] .* 100,        bar_position=:dodge,        bar_width=0.7,        xticks=(1:length(model_names_bal), model_names_bal),        xlabel="Model",        ylabel="Score (%)",        title="Model Comparison - BALANCED Dataset (50/50)",        label=["Sensitivity" "Precision" "F1 Score"],        legend=:best,        ylims=(0, 100),        size=(900, 500),        xrotation=45    )        display(p_bal)end# Prepare data for REALISTIC datasetif !isempty(realistic_models)    model_names_real = collect(keys(realistic_models))    sensitivities_real = [realistic_models[m]["sensitivity"] for m in model_names_real]    precisions_real = [realistic_models[m]["precision"] for m in model_names_real]    f1_scores_real = [realistic_models[m]["f1"] for m in model_names_real]        p_real = groupedbar(        [sensitivities_real precisions_real f1_scores_real] .* 100,        bar_position=:dodge,        bar_width=0.7,        xticks=(1:length(model_names_real), model_names_real),        xlabel="Model",        ylabel="Score (%)",        title="Model Comparison - REALISTIC Dataset (Unbalanced)",        label=["Sensitivity" "Precision" "F1 Score"],        legend=:best,        ylims=(0, 100),        size=(900, 500),        xrotation=45    )        display(p_real)endprintln("\n✅ Visualizations complete!")println("\nKey Observations:")println("• Balanced data typically shows higher precision")println("• Realistic data reveals true production challenges")println("• Sensitivity should remain high across both datasets")